# DCC 2026 Mission 3: KcBERT 다중 증상 분류

119 신고 대화의 전체 전사 텍스트를 입력으로 사용해 **9종 환자 증상**을 multi-label로 분류합니다.

- Training(서울) 데이터만 모델 학습과 내부 검증에 사용합니다.
- 공식 Validation은 최종 추론에만 사용해 데이터 누수를 막습니다.
- KcBERT의 `[CLS]` 표현을 9개의 logit으로 변환하고 `BCEWithLogitsLoss`로 학습합니다.
- 모든 코드는 이 Notebook 하나에 포함되어 별도 Python 모듈이 필요하지 않습니다.

## 0. 실행 전 확인

Notebook은 저장소 루트 또는 `MISSION3` 폴더에서 실행할 수 있습니다. 데이터가 자동으로 발견되지 않으면 아래 설정 셀의 `DATA_ROOT_OVERRIDE`에 `Data` 폴더 경로를 입력하세요.

처음 실행하는 환경에서는 다음 패키지가 필요합니다.

```python
%pip install torch numpy pandas transformers tqdm
```

## 1. 라이브러리와 고정 설정

재현에 필요한 모델명, seed, 학습 설정과 9개 증상 순서를 한곳에서 관리합니다. `TARGET_SYMPTOMS`의 순서는 학습 label과 제출 결과를 연결하는 기준이므로 변경하지 않습니다.

In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

# 데이터가 자동 탐색되지 않을 때만 문자열 경로를 입력합니다.
# 예: DATA_ROOT_OVERRIDE = r"C:\\path\\to\\Data"
DATA_ROOT_OVERRIDE: str | None = None

MODEL_NAME = "beomi/kcbert-base"
TARGET_SYMPTOMS = [
    "고열", "구토", "두통", "복통", "어지러움",
    "열상", "오심", "전신쇠약", "호흡곤란",
]

SEED = 42
MAX_LENGTH = 256
BATCH_SIZE = 24
EPOCHS = 3
LEARNING_RATE = 2e-5
INTERNAL_VALIDATION_RATIO = 0.1
PREDICTION_THRESHOLD = 0.5


## 2. 데이터 경로와 JSON 변환 함수

각 통화의 `utterances`에 있는 전사 문장을 순서대로 연결합니다. 원본 symptom 중 대회 평가 대상 9개만 남긴 뒤 고정 순서의 multi-hot vector로 변환합니다.

In [ ]:
def find_project_dir() -> Path:
    """현재 위치를 기준으로 Mission 3 작업 폴더를 찾습니다."""
    current = Path.cwd().resolve()
    if current.name.casefold() == "mission3":
        return current
    if (current / "MISSION3").is_dir():
        return current / "MISSION3"
    return current


def find_data_root(project_dir: Path) -> Path:
    """직접 지정값, 환경변수, 대표 상대경로 순서로 Data 폴더를 찾습니다."""
    candidates: list[Path] = []
    if DATA_ROOT_OVERRIDE:
        candidates.append(Path(DATA_ROOT_OVERRIDE))
    if os.environ.get("DCC_DATA_ROOT"):
        candidates.append(Path(os.environ["DCC_DATA_ROOT"]))
    candidates.extend([
        project_dir / "Data",
        project_dir / "data",
        project_dir.parent / "Data",
        project_dir.parent.parent / "예선데이터" / "Data",
    ])

    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if resolved.is_dir():
            return resolved
    raise FileNotFoundError(
        "Data 폴더를 찾지 못했습니다. DATA_ROOT_OVERRIDE 또는 "
        "환경변수 DCC_DATA_ROOT를 설정하세요."
    )


def resolve_label_dir(data_root: Path, patterns: list[str]) -> Path:
    """후보 glob 패턴 중 실제로 존재하는 첫 라벨 폴더를 반환합니다."""
    for pattern in patterns:
        matches = sorted(path for path in data_root.glob(pattern) if path.is_dir())
        if matches:
            return matches[0]
    raise FileNotFoundError(f"라벨 폴더를 찾지 못했습니다: {patterns}")


def load_label(label_path: Path) -> dict[str, Any]:
    """UTF-8 JSON 라벨 파일을 읽습니다."""
    with label_path.open(encoding="utf-8") as file:
        return json.load(file)


def list_label_files(label_dir: Path) -> list[Path]:
    """재현 가능한 순서를 위해 JSON 파일명을 정렬합니다."""
    return sorted(label_dir.glob("*.json"))


def build_dialogue_text(label: dict[str, Any]) -> str:
    """한 통화의 모든 발화 전사를 순서대로 공백 연결합니다."""
    utterance_texts = [
        utterance.get("text", "").strip()
        for utterance in label.get("utterances", [])
    ]
    return " ".join(text for text in utterance_texts if text)


def filter_symptoms(symptoms: list[str]) -> list[str]:
    """원본 증상 중 평가 대상 9개만 유지합니다."""
    target_set = set(TARGET_SYMPTOMS)
    return [symptom for symptom in symptoms if symptom in target_set]


def symptoms_to_multihot(symptoms: list[str]) -> list[int]:
    """증상 목록을 TARGET_SYMPTOMS 순서의 9차원 vector로 변환합니다."""
    selected = set(filter_symptoms(symptoms))
    return [int(symptom in selected) for symptom in TARGET_SYMPTOMS]


def multihot_to_symptom_str(multihot: list[int]) -> str:
    """예측 vector를 제출 CSV의 증상 문자열 형식으로 변환합니다."""
    selected = [
        symptom
        for symptom, value in zip(TARGET_SYMPTOMS, multihot)
        if value == 1
    ]
    return str(selected)


PROJECT_DIR = find_project_dir()
DATA_ROOT = find_data_root(PROJECT_DIR)
TRAIN_LABEL_DIR = resolve_label_dir(
    DATA_ROOT,
    ["training/2.*/TL_*", "Training/2.*/TL_*"],
)
VALIDATION_LABEL_DIR = resolve_label_dir(
    DATA_ROOT,
    ["Validation/2.*/VL_*", "validation/2.*/VL_*"],
)
CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "mission3.pt"
SUBMISSION_PATH = PROJECT_DIR / "outputs" / "mission3.csv"

print("PROJECT_DIR          :", PROJECT_DIR)
print("DATA_ROOT           :", DATA_ROOT)
print("TRAIN_LABEL_DIR     :", TRAIN_LABEL_DIR)
print("VALIDATION_LABEL_DIR:", VALIDATION_LABEL_DIR)


## 3. 데이터 계약 확인

학습 전에 파일 수, JSON 구조, 빈 대화, multi-label 변환을 확인합니다. 이 단계는 잘못된 폴더나 라벨 형식으로 긴 학습을 시작하는 실수를 방지합니다.

In [ ]:
training_label_files = list_label_files(TRAIN_LABEL_DIR)
validation_label_files = list_label_files(VALIDATION_LABEL_DIR)

assert training_label_files, "Training JSON 파일이 없습니다."
assert validation_label_files, "Validation JSON 파일이 없습니다."
assert not ({path.name for path in training_label_files} & {path.name for path in validation_label_files})

sample_label = load_label(training_label_files[0])
assert isinstance(sample_label.get("utterances", []), list)
assert isinstance(sample_label.get("symptom", []), list)
assert len(symptoms_to_multihot(sample_label.get("symptom", []))) == len(TARGET_SYMPTOMS)

symptom_counts = Counter()
empty_dialogue_count = 0
for label_path in training_label_files:
    label = load_label(label_path)
    empty_dialogue_count += int(not build_dialogue_text(label))
    symptom_counts.update(filter_symptoms(label.get("symptom", [])))

print(f"Training calls  : {len(training_label_files):,}")
print(f"Validation calls: {len(validation_label_files):,}")
print(f"Empty dialogues : {empty_dialogue_count:,}")
print("\nTraining label distribution")
for symptom in TARGET_SYMPTOMS:
    print(f"{symptom:8s}: {symptom_counts[symptom]:6,d}")


## 4. 핵심 EDA: 대화 길이·발화 수·토큰 길이

Training 전체를 기준으로 통화별 대화 문자 수, 발화 수, KcBERT token 수를 확인합니다. 특히 `MAX_LENGTH=256`을 넘는 비율은 모델 입력에서 잘리는 대화가 얼마나 많은지 보여줍니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dialogue_character_lengths = []
utterance_counts = []
token_lengths = []
target_label_counts = []

for label_path in tqdm(training_label_files, desc="EDA"):
    label = load_label(label_path)
    dialogue = build_dialogue_text(label)
    model_input = dialogue or "증상 없음"

    dialogue_character_lengths.append(len(dialogue))
    utterance_counts.append(len(label.get("utterances", [])))
    target_label_counts.append(len(filter_symptoms(label.get("symptom", []))))
    token_lengths.append(
        len(
            tokenizer(
                model_input,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )
    )


def summarize(values: list[int]) -> list[float]:
    """길이 분포를 평균, 중앙값, 95백분위, 최댓값으로 요약합니다."""
    array = np.asarray(values)
    return [
        float(array.mean()),
        float(np.median(array)),
        float(np.percentile(array, 95)),
        float(array.max()),
    ]


eda_summary = pd.DataFrame(
    {
        "대화 문자 수": summarize(dialogue_character_lengths),
        "발화 수": summarize(utterance_counts),
        "KcBERT token 수": summarize(token_lengths),
        "평가 대상 증상 수": summarize(target_label_counts),
    },
    index=["평균", "중앙값", "95백분위", "최댓값"],
).round(1)
display(eda_summary)

token_length_array = np.asarray(token_lengths)
for token_limit in [MAX_LENGTH, 512, 768]:
    over_limit = token_length_array > token_limit
    print(
        f"{token_limit:>3} token 초과: {over_limit.sum():,}건 "
        f"({over_limit.mean():.2%})"
    )

print(f"복수 증상 통화: {(np.asarray(target_label_counts) >= 2).sum():,}건")


## 5. Tokenizer와 Dataset

통화 1건을 샘플 1개로 사용합니다. 대화가 비어 있으면 `증상 없음`을 입력하고, 긴 대화는 KcBERT 입력 한도 안에서 앞쪽 256 token만 사용합니다. Training의 10%를 고정 seed로 떼어 내부 검증에 사용하며 공식 Validation은 이 분할에 포함하지 않습니다.

In [ ]:
def set_seed(seed: int) -> None:
    """Python, NumPy, PyTorch의 난수 seed를 고정합니다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def split_training_files(
    label_files: list[Path],
    validation_ratio: float,
    seed: int,
) -> tuple[list[Path], list[Path]]:
    """Training 파일을 재현 가능한 내부 train/validation으로 분할합니다."""
    shuffled_files = label_files.copy()
    random.Random(seed).shuffle(shuffled_files)
    validation_size = int(len(shuffled_files) * validation_ratio)
    return shuffled_files[validation_size:], shuffled_files[:validation_size]


class Mission3Dataset(Dataset):
    """통화 전사와 9차원 multi-hot label을 반환합니다."""

    def __init__(
        self,
        label_files: list[Path],
        tokenizer,
        max_length: int,
    ) -> None:
        self.label_files = label_files
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.label_files)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        label = load_label(self.label_files[index])
        dialogue = build_dialogue_text(label) or "증상 없음"
        encoded = self.tokenizer(
            dialogue,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        targets = torch.tensor(
            symptoms_to_multihot(label.get("symptom", [])),
            dtype=torch.float32,
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": targets,
        }


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    """Windows와 CUDA 사용 여부에 맞춰 DataLoader를 구성합니다."""
    worker_count = 0 if sys.platform == "win32" else 2
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=worker_count,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=worker_count > 0,
    )


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_files, internal_validation_files = split_training_files(
    training_label_files, INTERNAL_VALIDATION_RATIO, SEED
)

train_dataset = Mission3Dataset(train_files, tokenizer, MAX_LENGTH)
internal_validation_dataset = Mission3Dataset(
    internal_validation_files, tokenizer, MAX_LENGTH
)
train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True)
internal_validation_loader = make_loader(
    internal_validation_dataset, BATCH_SIZE, shuffle=False
)

print("Device             :", device)
print(f"Internal train     : {len(train_dataset):,}")
print(f"Internal validation: {len(internal_validation_dataset):,}")


## 6. KcBERT 증상 분류 모델

KcBERT의 첫 token(`[CLS]`) 표현을 dropout과 linear layer에 통과시켜 증상별 logit 9개를 출력합니다. 각 증상은 독립적으로 존재할 수 있으므로 softmax가 아닌 sigmoid 기반 multi-label 분류를 사용합니다.

In [ ]:
class SymptomClassifier(nn.Module):
    """전체 대화 전사에서 9종 증상의 logit을 예측합니다."""

    def __init__(self, model_name: str, num_labels: int) -> None:
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_embedding = self.dropout(outputs.last_hidden_state[:, 0])
        return self.classifier(cls_embedding)


## 7. 학습과 내부 검증

`BCEWithLogitsLoss`로 9개 label을 동시에 학습합니다. 내부 검증에서는 sigmoid 확률이 0.5보다 큰 증상을 선택하고, 증상별 F1의 평균인 Macro F1을 계산합니다. 가장 높은 Macro F1의 가중치만 저장합니다.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    scaler,
) -> float:
    """한 epoch을 학습하고 평균 loss를 반환합니다."""
    model.train()
    total_loss = 0.0
    use_amp = scaler is not None and device.type == "cuda"

    for batch in tqdm(loader, desc="train", leave=False):
        optimizer.zero_grad(set_to_none=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


def macro_f1_by_label(
    labels: np.ndarray, predictions: np.ndarray
) -> tuple[float, np.ndarray]:
    """증상별 F1과 그 macro 평균을 계산합니다."""
    true_positive = ((predictions == 1) & (labels == 1)).sum(axis=0)
    false_positive = ((predictions == 1) & (labels == 0)).sum(axis=0)
    false_negative = ((predictions == 0) & (labels == 1)).sum(axis=0)

    precision = true_positive / (true_positive + false_positive + 1e-8)
    recall = true_positive / (true_positive + false_negative + 1e-8)
    label_f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return float(label_f1.mean()), label_f1


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
) -> tuple[float, np.ndarray]:
    """내부 검증 데이터의 Macro F1과 증상별 F1을 반환합니다."""
    model.eval()
    prediction_batches = []
    label_batches = []

    for batch in tqdm(loader, desc="eval", leave=False):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        logits = model(input_ids, attention_mask)
        predictions = (torch.sigmoid(logits) > threshold).int().cpu().numpy()
        prediction_batches.append(predictions)
        label_batches.append(batch["labels"].int().numpy())

    all_predictions = np.vstack(prediction_batches)
    all_labels = np.vstack(label_batches)
    return macro_f1_by_label(all_labels, all_predictions)


def save_checkpoint(
    path: Path,
    model: nn.Module,
    epoch: int,
    best_f1: float,
) -> None:
    """최고 성능 모델과 재현에 필요한 정보를 저장합니다."""
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "mission": 3,
            "epoch": epoch,
            "best_f1": best_f1,
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "threshold": PREDICTION_THRESHOLD,
            "target_symptoms": TARGET_SYMPTOMS,
        },
        path,
    )


In [ ]:
model = SymptomClassifier(MODEL_NAME, len(TARGET_SYMPTOMS)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
best_macro_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler
    )
    macro_f1, label_f1 = evaluate(
        model, internal_validation_loader, device, PREDICTION_THRESHOLD
    )
    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"loss={train_loss:.4f} | macro_f1={macro_f1:.4f}"
    )
    print(
        "  " + " | ".join(
            f"{symptom}={score:.3f}"
            for symptom, score in zip(TARGET_SYMPTOMS, label_f1)
        )
    )

    if macro_f1 >= best_macro_f1:
        best_macro_f1 = macro_f1
        save_checkpoint(CHECKPOINT_PATH, model, epoch, best_macro_f1)
        print(f"  Best checkpoint saved: {CHECKPOINT_PATH}")

print(f"Best internal validation Macro F1: {best_macro_f1:.4f}")


## 8. 공식 Validation 추론

내부 검증 점수가 가장 높았던 체크포인트를 다시 불러와 공식 Validation 3,640건을 batch 단위로 추론합니다. 예측된 증상은 원래 JSON 파일명과 함께 대회 제출 형식으로 저장합니다.

In [ ]:
class Mission3InferenceDataset(Dataset):
    """공식 Validation의 파일명과 tokenized dialogue를 반환합니다."""

    def __init__(
        self,
        label_files: list[Path],
        tokenizer,
        max_length: int,
    ) -> None:
        self.label_files = label_files
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.label_files)

    def __getitem__(self, index: int) -> dict[str, Any]:
        label_path = self.label_files[index]
        dialogue = build_dialogue_text(load_label(label_path)) or "증상 없음"
        encoded = self.tokenizer(
            dialogue,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "file_name": label_path.name,
        }


@torch.no_grad()
def predict_validation(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
) -> pd.DataFrame:
    """공식 Validation 예측을 제출 형식의 DataFrame으로 반환합니다."""
    model.eval()
    rows: list[dict[str, str]] = []

    for batch in tqdm(loader, desc="inference"):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        logits = model(input_ids, attention_mask)
        predictions = (torch.sigmoid(logits) > threshold).int().cpu().tolist()

        rows.extend(
            {
                "label file name": file_name,
                "symptom": multihot_to_symptom_str(prediction),
            }
            for file_name, prediction in zip(batch["file_name"], predictions)
        )

    return pd.DataFrame(rows, columns=["label file name", "symptom"])


checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
assert checkpoint.get("target_symptoms", TARGET_SYMPTOMS) == TARGET_SYMPTOMS
inference_model_name = checkpoint.get("model_name", MODEL_NAME)
inference_max_length = checkpoint.get("max_length", MAX_LENGTH)
inference_threshold = checkpoint.get("threshold", PREDICTION_THRESHOLD)
inference_tokenizer = AutoTokenizer.from_pretrained(inference_model_name)
inference_model = SymptomClassifier(
    inference_model_name, len(TARGET_SYMPTOMS)
).to(device)
inference_model.load_state_dict(checkpoint["model_state_dict"])

inference_dataset = Mission3InferenceDataset(
    validation_label_files, inference_tokenizer, inference_max_length
)
inference_loader = make_loader(
    inference_dataset, BATCH_SIZE, shuffle=False
)
submission = predict_validation(
    inference_model, inference_loader, device, inference_threshold
)


## 9. 제출 파일 저장과 최종 검사

행 수, 컬럼 순서, 파일명 중복 여부를 확인한 뒤 한글이 깨지지 않도록 UTF-8 BOM 형식으로 CSV를 저장합니다.

In [ ]:
expected_columns = ["label file name", "symptom"]
assert submission.columns.tolist() == expected_columns
assert len(submission) == len(validation_label_files)
assert submission["label file name"].is_unique
assert submission["label file name"].str.endswith(".json").all()

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8-sig")

display(submission.head())
print(f"Mission 3 submission: {len(submission):,} rows -> {SUBMISSION_PATH}")
